# Objective: Build and evaluate a linear regression model that predicts house prices based on features such as area, location, number of rooms, and age. Develop end-to-end skills from data cleaning through to model interpretation.

### Tech Stack: Python, pandas, scikit-learn, matplotlib, seaborn, Jupyter Notebook 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
    #loading the dataset
df=pd.read_csv("data\Housing.csv")

In [ ]:
#rows and columns
df.shape

In [ ]:
#columns
df.columns

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.info()

In [ ]:
#checking null values
df.isnull().any()

No null values!

In [ ]:
#checking duplicated rows
df.duplicated().any()

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(df['price'],
             bins=30,
             kde=True)

plt.title("Distribution of House Prices")
plt.savefig('images/price_distribution.png')
plt.show()

The target variable (price) is positively skewed. Most houses are concentrated in the lower and medium price ranges, while only a few luxury houses have very high prices. These high-priced properties create a long tail on the right side of the distribution, making the data right-skewed.

In [ ]:
num_cols=['price','area','bedrooms','bathrooms','stories','parking']

df[num_cols].hist(figsize=(15,10), bins=20)
plt.savefig('images/num_cols_distribution.png')
plt.tight_layout()

- Price: The distribution is right-skewed, indicating that most houses are priced in the lower to mid-range, while a few - luxury houses have very high prices.
- Area: Most houses have relatively smaller living areas, with only a few properties having very large areas, resulting in a right-skewed distribution.
- Bedrooms: The majority of houses have 2 to 4 bedrooms, with very few houses having a larger number of bedrooms.
- Bathrooms: Most houses contain 1 or 2 bathrooms, while houses with more bathrooms are less common.
- Stories: Houses with 1 or 2 stories are the most frequent, whereas multi-storey houses (3 or more stories) appear less often.
- Parking: Most properties provide 0 to 2 parking spaces, and only a small number of houses offer more parking capacity.

## Feature selection



The following features are expected to influence house prices:

- Area: Larger houses generally cost more.
- Bedrooms: More bedrooms often indicate larger homes.
- Bathrooms: Houses with more bathrooms usually have higher market value.
- Stories: Multi-storey houses are often priced higher.
- Parking: Additional parking spaces increase convenience and value.
- Main Road: Accessibility can positively affect prices.
- Guest Room: An extra guest room may increase desirability.
- Basement: Additional usable space can increase value.
- Air Conditioning: Improves comfort and resale value.
- Preferred Area: Houses in preferred neighbourhoods generally command higher prices.
- Furnishing Status: Furnished homes are usually sold at a premium

In [ ]:
#categorical columns
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
plt.figure(figsize=(12,8))
sns.heatmap(df_encoded.corr(),
            annot=True,
            cmap='coolwarm',
            fmt=".2f")

plt.title("Correlation Heatmap")
plt.savefig('images/corr_heatmap.png')
plt.show()

- Area has the strongest positive correlation with price, indicating that houses with a larger area generally have higher selling prices.
- Bathrooms and stories also show a moderate positive correlation with price, suggesting that houses with more bathrooms and additional floors tend to be more valuable.
- Air Conditioning (airconditioning_yes) is positively correlated with price, meaning houses equipped with air conditioning are generally sold at higher prices.
- Parking has a positive correlation with house price, indicating that additional parking spaces add value to a property.
- Preferred Area (prefarea_yes) shows a positive relationship with price, suggesting that houses located in preferred residential areas are typically more expensive.
- Furnished (furnishingstatus_furnished) houses generally have a positive correlation with price, whereas unfurnished houses tend to have a comparatively lower association with higher prices.
- Features such as mainroad, guestroom, basement, and hotwaterheating exhibit relatively weak positive correlations with price, indicating that they may contribute to house value but have a smaller individual impact.

## Training

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

In [ ]:
categorical_cols = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ],
    remainder='passthrough'
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

## Error evaluation

In [ ]:
mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)

The Mean Squared Error (MSE) represents the average of the squared differences between the actual and predicted house prices. Since the errors are squared, larger prediction errors receive a greater penalty. The high numerical value is expected because house prices are measured in large monetary units.

In [ ]:
rmse = np.sqrt(mse)
print("RMSE:", rmse)

The Root Mean Squared Error (RMSE) is approximately 1.32 million, meaning that, on average, the model's predicted house prices differ from the actual prices by about 1.32 million units of currency. RMSE is expressed in the same unit as the target variable, making it easier to interpret than MSE.

In [ ]:
r2 = r2_score(y_test, y_pred)
print("R² Score:", r2)

The R² Score of 0.653 indicates that the Linear Regression model explains approximately 65.3% of the variation in house prices based on the selected features. This suggests that the model has moderate predictive performance, while the remaining 34.7% of the variation may be due to factors not included in the dataset or inherent randomness.

### Actual vs Predicted scatter plot

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y_test, y_pred)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red'
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Prices")
plt.savefig('images/actual_vs_pred_scatter.png')
plt.show()

- A large number of data points are clustered close to the red diagonal line, indicating that the model's predictions are reasonably close to the actual house prices.
- Points lying directly on or very near the diagonal line represent highly accurate predictions with minimal error.
- Some points are scattered farther away from the diagonal, indicating instances where the model either overestimated or underestimated the house prices.
- Although a few prediction errors exist, the overall distribution of points suggests that the model captures the general relationship between the input features and house prices effectively.

### Residual Plot

In [ ]:
residuals= y_test - y_pred
plt.figure(figsize=(8,5))

sns.scatterplot(x=y_pred, y=residuals)

plt.axhline(0, color='red')

plt.xlabel("Predicted Prices")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.savefig('images/residual_plot.png')
plt.show()

- The residuals are distributed around the horizontal reference line at 0, indicating that the prediction errors include both positive and negative values.
- Most residuals appear to be randomly scattered without any strong systematic pattern.
- A few observations lie farther from the zero line, representing houses where the prediction error is relatively larger.
- The absence of a clear trend suggests that the Linear Regression model provides a reasonable fit for the dataset.

### Coefficient Analysis

In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

coefficients = model.named_steps['regressor'].coef_

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

coef_df = coef_df.sort_values(by='Coefficient', ascending=False)
coef_df

- Features with the largest positive coefficients have the strongest positive impact on house prices and contribute to increasing the predicted value.
- Features with the largest negative coefficients have the strongest negative impact on house prices and contribute to decreasing the predicted value.
- The coefficient values help identify which house characteristics are most influential in the Linear Regression model.

In [ ]:
plt.figure(figsize=(10,8))

sns.barplot(
    data=coef_df,
    x='Coefficient',
    y='Feature'
)

plt.title("Feature Importance (Linear Regression Coefficients)")
plt.savefig('images/coefficients.png')
plt.show()

- Bathrooms has the highest positive coefficient (≈ 1,094,445), making it the most influential predictor in the model. Houses with more bathrooms are predicted to have significantly higher prices.
- Air Conditioning (≈ 791,427) and Hot Water Heating (≈ 684,650) also have strong positive coefficients, suggesting that these amenities considerably increase a property's value.
- Houses located in a Preferred Area (≈ 629,891) are predicted to be more expensive than those in non-preferred locations.
- Stories (≈ 407,477) and the presence of a Basement (≈ 390,251) positively influence house prices, indicating that larger or more spacious homes generally command higher values.
- Houses connected to the Main Road (≈ 367,920) and those with a Guest Room (≈ 231,610) also contribute positively to the predicted selling price.
- Parking (≈ 224,842) has a positive impact, showing that additional parking spaces increase the property's market value.
- Bedrooms (≈ 76,779) have a relatively smaller positive effect compared to bathrooms and other amenities.
- Area has a positive coefficient (≈ 236), indicating that each additional unit of area contributes positively to the house price. Although the coefficient appears numerically small, it is measured per unit of area, so the cumulative impact becomes substantial for larger properties.
### Negative Coefficients
- Semi-furnished houses have a negative coefficient (≈ -126,882), indicating that they are predicted to be priced lower than the reference category used during One-Hot Encoding.
- Unfurnished houses have the largest negative coefficient (≈ -413,645), suggesting that they are generally valued significantly lower than the reference furnishing category.

### Ridge Regression

In [ ]:
ridge_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)

ridge_pred = ridge_model.predict(X_test)

ridge_r2 = r2_score(y_test, ridge_pred)

print("Ridge R²:", ridge_r2)

### Lasso regression

In [ ]:
lasso_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=1.0))
])

lasso_model.fit(X_train, y_train)

lasso_pred = lasso_model.predict(X_test)

lasso_r2 = r2_score(y_test, lasso_pred)

print("Lasso R²:", lasso_r2)

### Comparing models

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso'],
    'R² Score': [
        r2_score(y_test, y_pred),
        r2_score(y_test, ridge_pred),
        r2_score(y_test, lasso_pred)
    ]
})

comparison

In [ ]:
plt.figure(figsize=(8,4))

sns.barplot(
    data=comparison,
    x='R² Score',
    y='Model',
    palette='viridis'
)

plt.title("Comparison of Regression Models (R² Score)")
plt.xlabel("R² Score")
plt.xlim(0,1)
plt.savefig('images/model_comparison.png')
plt.show()


- All three models achieved very similar R² scores, with values close to 0.65.
- The Linear Regression and Lasso Regression models produced almost identical performance, while Ridge Regression performed only marginally lower.
- The nearly equal bar lengths indicate that applying L1 (Lasso) or L2 (Ridge) regularization did not produce a significant improvement in the model's predictive ability.

The similarity in performance suggests that the original Linear Regression model is already stable and does not suffer from severe overfitting or multicollinearity. As a result, adding regularization has only a minimal effect on the model's ability to predict house prices.